In [1]:
## SET SOME PATHS -- YOU MAY NEED TO CHANGE THIS CELL
import os

# ensure we are in the project root
%cd "/mnt/ps/home/CORP/charlie.jones/project/big-img/"

# where we will save our parquet files
SAVE_DIR = "/mnt/ps/home/CORP/charlie.jones/project/big-img/metadata"
os.makedirs(SAVE_DIR, exist_ok=True)

/rxrx/data/user/charlie.jones/big-img


In [2]:
import polars as pl

def preprocess_df(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    # any data without a path to an image is useless
    num_to_drop = df.filter(pl.col("path").is_null()).shape[0]
    print(f"Dropping {num_to_drop} rows with null paths")
    df = df.filter(~pl.col("path").is_null())

    # drop wells with shape != [2048, 2048, 6]
    num_to_drop = df.filter(pl.col("shape") != [2048, 2048, 6]).shape[0]
    print(f"Dropping {num_to_drop:,} rows with shape != [2048, 2048, 6]")
    df = df.filter(pl.col("shape") == [2048, 2048, 6])

    # well_id uniquely identifies wells (rows) in the dataset
    df = df.with_columns(
        pl.concat_str(
            [pl.col("site"), pl.col("experiment"), pl.col("plate"), pl.col("address")]
        ).alias("well_id")
    )

    # create empty control col when rec_id == []
    df = df.with_columns(
        pl.when(pl.col("rec_id").list.len() == 0)
        .then(True)
        .otherwise(False)
        .alias("empty_control")
    )

    # drop any experiments with <N empty wells or <N perturbed wells
    experiment_counts = df.group_by("experiment").agg(
        pl.col("empty_control").sum().alias("n_empty_wells"),
        (~pl.col("empty_control")).sum().alias("n_perturbed_wells"),
    )
    experiments_to_drop = experiment_counts.filter(
        (pl.col("n_empty_wells") < 250) | (pl.col("n_perturbed_wells") < 250)
    ).select("experiment")

    print(
        f"Dropping {len(experiments_to_drop)} experiments with less than 250 empty and perturbed wells."
    )
    df = df.filter(~df["experiment"].is_in(pl.Series(experiments_to_drop)))

    # only keep the columns we need
    df = df.select(
        [
            "well_id",            # unique ID of the well (image)
            "rec_id",             # string ID of the gene+guide combination
            "perturbation",       # string ID of the gene+guide+concentration combination
            "site",               # integer ID of where the data was collected
            "experiment",         # string ID of the experiment (biological batch)
            "plate",              # integer ID of the plate in the experiment
            "address",            # string ID of the well on the plate
            "perturbation_type",  # type of perturbation
            "cell_type",          # type of the cell e.g. HUVEC
            "image_type",         # whether images are BF,CP etc.
            "concentration",      # concentration of the compound (if applicable)
            "gene",               # gene being targeted (if applicable)
            "path",               # path to the image
            "empty_control",      # whether the well is an empty control
        ]
    )

    # check for duplicate rows (where all metadata in the selected columns is identical)
    n_to_drop = len(df) - len(df.unique())
    print(f"Dropping {n_to_drop} duplicate rows with identical metadata")
    df = df.unique()
    print(f"Remaining wells: {df.shape[0]:,}")
    print(f"{df.filter(df['empty_control']).shape[0]:,} empty wells")
    print(f"{df.filter(~df['empty_control']).shape[0]:,} perturbed wells")
    return df

In [3]:

df = pl.read_parquet('/rxrx/data/microscopy/photosynthetic_metadata/95M_filtered_6chan_rp006.parquet')
df = preprocess_df(df)

print(f"{df["perturbation"].unique().shape[0]:,} unique perturbations")
print(f"{df["experiment"].unique().shape[0]:,} unique experiments")
print(f"{df["cell_type"].unique().shape[0]:,} unique cell types")

print(df["perturbation_type"].value_counts(sort=True))
print(df["image_type"].value_counts(sort=True))
print(df["cell_type"].value_counts(sort=True))


Dropping 0 rows with null paths
Dropping 33,472,396 rows with shape != [2048, 2048, 6]
Dropping 5055 experiments with less than 250 empty and perturbed wells.


/tmp/ipykernel_983110/13355797.py:41: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.filter(~df["experiment"].is_in(pl.Series(experiments_to_drop)))


Dropping 6060 duplicate rows with identical metadata
Remaining wells: 47,375,251
3,439,006 empty wells
43,936,245 perturbed wells
2,308,872 unique perturbations
3,016 unique experiments
21 unique cell types
shape: (51, 2)
┌─────────────────────────────────┬──────────┐
│ perturbation_type               ┆ count    │
│ ---                             ┆ ---      │
│ list[str]                       ┆ u32      │
╞═════════════════════════════════╪══════════╡
│ ["GUIDE"]                       ┆ 18833691 │
│ ["COMPOUND", "GUIDE"]           ┆ 17314473 │
│ ["COMPOUND"]                    ┆ 4707693  │
│ []                              ┆ 3439006  │
│ ["COMPOUND", "SOLUBLE_FACTOR"]  ┆ 1228021  │
│ …                               ┆ …        │
│ ["EMPTY", "EMPTY", … "GUIDE"]   ┆ 59       │
│ ["COMPOUND", "ANTIBODY"]        ┆ 56       │
│ ["COMPOUND", "COMPOUND", "SOLU… ┆ 54       │
│ ["GUIDE", "GUIDE", … "GUIDE"]   ┆ 9        │
│ ["EMPTY", "EMPTY", "EMPTY"]     ┆ 1        │
└─────────────────────────

In [4]:
import numpy as np

# create iid_cellpaint and iid_brightfield splits, each with 32768 rows
cellpaint_df = df.filter(pl.col("image_type") == "cellpaint")
brightfield_df = df.filter(pl.col("image_type").is_in(["brightfield_live", "brightfield_fixed"]))

def create_paired_validation_split(
    df: pl.DataFrame,
    n_samples: int = 32768,
    seed: int = 42
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Creates a validation and training split from a high-throughput screening dataframe.

    The function samples a specified number of perturbation wells and pairs each one
    with a randomly selected control well from the same experiment.

    Args:
        df (pl.DataFrame): The input dataframe. Must contain the columns 'experiment' (str)
                           and 'empty_control' (bool).
        n_samples (int, optional): The number of perturbation wells (and control wells)
                                   to include in the validation split. Defaults to 32768.
        seed (int, optional): A random seed for reproducibility. Defaults to 42.

    Returns:
        tuple[pl.DataFrame, pl.DataFrame]: A tuple containing the validation dataframe
                                           and the training dataframe.
    """
    # Set the seed for reproducibility
    np.random.seed(seed)

    # Add a unique identifier to each well for easy tracking
    if 'id' not in df.columns:
        df_with_id = df.with_row_count(name="id")
    else:
        # Ensure the ID is unique if it already exists
        df_with_id = df.with_columns(pl.col('id').cast(str) + "_" + pl.arange(0, len(df)).cast(str))


    # 1. Separate perturbation and control wells
    pert_df = df_with_id.filter(pl.col('empty_control') == False)
    ctrl_df = df_with_id.filter(pl.col('empty_control') == True)

    # 2. Randomly sample the desired number of perturbation wells for the validation set
    validation_pert = pert_df.sample(n=n_samples, with_replacement=False, shuffle=True, seed=seed)

    # 3. Count how many validation perturbations were sampled from each experiment
    pert_counts_per_experiment = validation_pert.group_by('experiment').agg(
        pl.len().alias('n_perts_for_validation')
    )

    # 4. Use the counts to sample a corresponding number of control wells from each experiment
    # First, filter the controls to only those in experiments we need
    joined_ctrl_df = ctrl_df.join(
        pert_counts_per_experiment, on='experiment', how='inner'
    )
    
    # Then, perform the random sampling within each group
    validation_ctrl = joined_ctrl_df.with_columns(
        # Add a random value to each row for shuffling within groups.
        # pl.rand() is only available in newer polars versions.
        # This creates a Series of the correct length for backward compatibility.
        pl.Series("random", np.random.rand(len(joined_ctrl_df)))
    ).with_columns(
        # Rank controls randomly within each experiment group
        pl.col('random').rank(method='random', seed=seed).over('experiment').alias('rank')
    ).filter(
        # Keep only the top N controls, where N is the number of perts from that experiment
        pl.col('rank') <= pl.col('n_perts_for_validation')
    ).drop(['random', 'rank', 'n_perts_for_validation'])


    # 5. Combine the sampled perturbation and control wells into the validation set
    validation_df = pl.concat([validation_pert, validation_ctrl])

    # 6. The training set is everything not in the validation set.
    # We use an anti-join on the unique 'id'.
    train_df = df_with_id.join(validation_df.select('id'), on='id', how='anti')

    # Remove the temporary ID column for cleanliness
    validation_df = validation_df.drop('id')
    train_df = train_df.drop('id')

    return train_df, validation_df

cp_train_df, cp_val_df = create_paired_validation_split(cellpaint_df)
bf_train_df, bf_val_df = create_paired_validation_split(brightfield_df)

train_df = pl.concat([cp_train_df, bf_train_df])


/tmp/ipykernel_983110/3505663959.py:34: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  df_with_id = df.with_row_count(name="id")


In [5]:
train_df.write_parquet(os.path.join(SAVE_DIR, "gigacell_train.parquet"))
cp_val_df.write_parquet(os.path.join(SAVE_DIR, "gigacell_iid_cp.parquet"))
bf_val_df.write_parquet(os.path.join(SAVE_DIR, "gigacell_iid_bf.parquet"))